In [7]:
import pandas as pd
import numpy as np

In [8]:
!pip3 install influxdb-client

In [9]:
import influxdb_client
import os
from influxdb_client import InfluxDBClient
from influxdb_client.client.query_api import QueryApi

# Configuração
token = "KRM9VOcRhSwGpHpSwhN55ICAG-GdtHtKftgIh1Wch4v18EQJpfludhC8fCaxcad1cm4xJ-nQsHgTJnuLqyfjbg=="
org = "feira"
url = "https://tsdb.feira-de-jogos.dev.br"
bucket = "feira"  # Substitua pelo nome do seu bucket

# Cliente
client = InfluxDBClient(url=url, token=token, org=org)
query_api = client.query_api()

In [10]:
def analisar_temperatura(data_inicio, data_fim, janela_minutos):
    """
    Analisa dados de temperatura em uma janela de tempo específica

    Args:
        data_inicio: string no formato "2025-08-21T11:00:00Z"
        data_fim: string no formato "2025-08-21T11:02:00Z"
        janela_minutos: duração da janela em minutos (ex: 2, 3, 5)
    """

    # Query para obter estatísticas dos dados de temperatura
    query = f"""
    from(bucket: "{bucket}")
      |> range(start: {data_inicio}, stop: {data_fim})
      |> filter(fn: (r) => r._field == "temp")
      |> aggregateWindow(every: {janela_minutos}m, fn: mean, createEmpty: false)
      |> yield(name: "media")
    """

    # Query para contagem de pontos
    query_count = f"""
    from(bucket: "{bucket}")
      |> range(start: {data_inicio}, stop: {data_fim})
      |> filter(fn: (r) => r._field == "temp")
      |> count()
      |> yield(name: "contagem")
    """

    # Query para média
    query_mean = f"""
    from(bucket: "{bucket}")
      |> range(start: {data_inicio}, stop: {data_fim})
      |> filter(fn: (r) => r._field == "temp")
      |> mean()
      |> yield(name: "media")
    """

    # Query para máximo
    query_max = f"""
    from(bucket: "{bucket}")
      |> range(start: {data_inicio}, stop: {data_fim})
      |> filter(fn: (r) => r._field == "temp")
      |> max()
      |> yield(name: "maximo")
    """

    # Query para mínimo
    query_min = f"""
    from(bucket: "{bucket}")
      |> range(start: {data_inicio}, stop: {data_fim})
      |> filter(fn: (r) => r._field == "temp")
      |> min()
      |> yield(name: "minimo")
    """

    # Query para desvio padrão
    query_stddev = f"""
    from(bucket: "{bucket}")
      |> range(start: {data_inicio}, stop: {data_fim})
      |> filter(fn: (r) => r._field == "temp")
      |> stddev()
      |> yield(name: "desvio_padrao")
    """

    print(f"\n{'='*60}")
    print(f"Análise de Temperatura - Janela de {janela_minutos} minutos")
    print(f"Período: {data_inicio} até {data_fim}")
    print(f"{'='*60}\n")

    # Executar queries
    try:
        # Contagem
        tables = query_api.query(query_count, org=org)
        for table in tables:
            for record in table.records:
                print(f"📊 Quantidade de dados coletados: {record.get_value()}")

        # Média
        tables = query_api.query(query_mean, org=org)
        for table in tables:
            for record in table.records:
                print(f"📈 Média da temperatura: {record.get_value():.2f}")

        # Máximo
        tables = query_api.query(query_max, org=org)
        for table in tables:
            for record in table.records:
                print(f"🔺 Temperatura máxima: {record.get_value():.2f}")

        # Mínimo
        tables = query_api.query(query_min, org=org)
        for table in tables:
            for record in table.records:
                print(f"🔻 Temperatura mínima: {record.get_value():.2f}")

        # Desvio padrão
        tables = query_api.query(query_stddev, org=org)
        for table in tables:
            for record in table.records:
                print(f"📉 Desvio padrão: {record.get_value():.2f}")

    except Exception as e:
        print(f"❌ Erro ao executar query: {e}")

    print(f"\n{'='*60}\n")


In [11]:
# EXEMPLOS DE USO:

# Análise de 2 minutos (11:00 às 11:02)
analisar_temperatura(
    data_inicio="2025-08-21T11:00:00Z",
    data_fim="2025-08-21T11:02:00Z",
    janela_minutos=2
)

# Análise de 3 minutos (11:00 às 11:03)
analisar_temperatura(
    data_inicio="2025-08-21T11:00:00Z",
    data_fim="2025-08-21T11:03:00Z",
    janela_minutos=3
)

# Análise de 5 minutos (11:00 às 11:05)
analisar_temperatura(
    data_inicio="2025-08-21T11:00:00Z",
    data_fim="2025-08-21T11:05:00Z",
    janela_minutos=5
)


Análise de Temperatura - Janela de 2 minutos
Período: 2025-08-21T11:00:00Z até 2025-08-21T11:02:00Z




Análise de Temperatura - Janela de 3 minutos
Período: 2025-08-21T11:00:00Z até 2025-08-21T11:03:00Z




Análise de Temperatura - Janela de 5 minutos
Período: 2025-08-21T11:00:00Z até 2025-08-21T11:05:00Z



